# Similarity Analysis — Varun Khedkar
Runs all 8 classifiers across 121 datasets, computes pairwise similarity matrices, and saves results to `similarity_results/` for download.

In [ ]:
# Step 1: Clone the repo, pull LFS data, install RAPIDS cuML for GPU acceleration
!apt-get install -y git-lfs -q
!git lfs install
!git clone https://github.com/MasksDemon/DS_Spring_2500_ML.git
%cd DS_Spring_2500_ML
!git lfs pull

# Install RAPIDS cuML (GPU-accelerated sklearn)
!pip install cuml-cu12 -q

In [ ]:
# Step 2: Verify GPU is available
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print("NOTE: Make sure you selected a GPU runtime: Runtime → Change runtime type → T4 GPU")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# GPU-accelerated classifiers (RAPIDS cuML)
from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.svm import SVC as cuSVC
from cuml.neighbors import KNeighborsClassifier as cuKNN
from cuml.linear_model import LogisticRegression as cuLogReg

# These don't have cuML equivalents — stay on sklearn (CPU)
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier

CLASSIFIERS = {
    'RandomForest': cuRF(n_estimators=100, random_state=42),
    'SVM':          cuSVC(kernel='rbf'),
    'NeuralNet':    MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'KNN':          cuKNN(n_neighbors=5),
    'NaiveBayes':   GaussianNB(),
    'AdaBoost':     AdaBoostClassifier(n_estimators=100, random_state=42),
    'LogReg':       cuLogReg(max_iter=1000),
}

DATA_DIR = Path('data-20260323T043051Z-3-001/data')
OUT_DIR  = Path('similarity_results')
OUT_DIR.mkdir(exist_ok=True)

print('Setup complete — GPU classifiers: RandomForest, SVM, KNN, LogReg')
print('CPU classifiers: NeuralNet, DecisionTree, NaiveBayes, AdaBoost')

In [ ]:
# Step 3: Build performance matrix
from cuml.model_selection import cross_val_score as cu_cross_val_score
from sklearn.model_selection import cross_val_score as sk_cross_val_score

GPU_MODELS = {'RandomForest', 'SVM', 'KNN', 'LogReg'}

dataset_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
total = len(dataset_dirs)
results = {}

for i, folder in enumerate(dataset_dirs, 1):
    name = folder.name
    vb = folder / f'{name}_version_b.csv'
    if not vb.exists():
        print(f'[{i}/{total}] {name}: skipped (no version_b)')
        continue
    try:
        df = pd.read_csv(vb)
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = LabelEncoder().fit_transform(df.iloc[:, -1].astype(str))
    except Exception as e:
        print(f'[{i}/{total}] {name}: skipped (load error: {e})')
        continue

    unique, counts = np.unique(y, return_counts=True)
    if counts.min() < 5:
        print(f'[{i}/{total}] {name}: skipped (too few samples in a class)')
        continue

    row = {}
    for clf_name, clf in CLASSIFIERS.items():
        try:
            if clf_name in GPU_MODELS:
                scores = cu_cross_val_score(clf, X, y, cv=5, scoring='accuracy')
            else:
                scores = sk_cross_val_score(clf, X, y, cv=5, scoring='accuracy')
            row[clf_name] = round(float(np.mean(scores)), 4)
        except Exception as e:
            row[clf_name] = np.nan
    results[name] = row
    print(f'[{i}/{total}] {name}: done')

df_performance = pd.DataFrame.from_dict(results, orient='index').dropna()
print(f'\nFinal matrix: {df_performance.shape[0]} datasets x {df_performance.shape[1]} models')
df_performance

In [ ]:
# Step 4: Compute similarity matrices
corr_matrix = df_performance.corr(method='pearson')

cos_array = cosine_similarity(df_performance.T)
cos_matrix = pd.DataFrame(cos_array, index=df_performance.columns, columns=df_performance.columns)

euc_array = euclidean_distances(df_performance.T)
euc_matrix = pd.DataFrame(euc_array, index=df_performance.columns, columns=df_performance.columns)

print('Pearson Correlation:')
display(corr_matrix.round(3))

print('\nCosine Similarity:')
display(cos_matrix.round(3))

print('\nEuclidean Distance:')
display(euc_matrix.round(3))

In [ ]:
# Step 5: Save all results to similarity_results/
df_performance.to_csv(OUT_DIR / 'model_performance_matrix.csv')
corr_matrix.to_csv(OUT_DIR / 'model_correlation_matrix.csv')
cos_matrix.to_csv(OUT_DIR / 'model_cosine_similarity_matrix.csv')
euc_matrix.to_csv(OUT_DIR / 'model_euclidean_distance_matrix.csv')

print('Saved to similarity_results/:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')

In [ ]:
# Step 6: Zip and download
import shutil
from google.colab import files

zip_path = shutil.make_archive('similarity_results', 'zip', '.', 'similarity_results')
print(f'Zipped: {zip_path}')
files.download('similarity_results.zip')